# 搭建一个Adaptive-RAG

In [13]:
import os
from langchain.chat_models import ChatOpenAI
from langchain.schema import AIMessage, HumanMessage, SystemMessage, ChatMessage
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredWordDocumentLoader, CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import Dict, List, Optional, Tuple, Union

import warnings

warnings.filterwarnings("ignore")

### LLM & Embedding 模型加载

In [1]:
from config import config
from llm_utils import SiliconFlowEmbedding, SiliconFlowChat

In [3]:
config.llm_api_key, config.emb_model, config.emb_api_base

('sk-mjlmettgpyvcjzadwauiesmzukmkfkrgxtlpbgcotzbtgicj',
 'BAAI/bge-large-zh-v1.5',
 'https://api.siliconflow.cn/v1/embeddings')

In [4]:
# 测试嵌入模型
emb_client = SiliconFlowEmbedding(api_key=config.llm_api_key,
                                           emb_model= config.emb_model,
                                           emb_api_url=config.emb_api_base)

test_texts = ["这是一个测试文档", "这是另一个测试文档"]
embeddings = emb_client.embed_documents(test_texts)
print(f"生成 {len(embeddings)} 个嵌入向量，每个维度为 {len(embeddings[0])}")

# 测试对话功能
chat_client = SiliconFlowChat(api_key=config.llm_api_key,
                                           chat_model= config.llm_model,
                                           chat_api_url= config.llm_api_base
                              )
messages = [
    {"role": "system", "content": "你是一个专业的AI助手。"},
    {"role": "user", "content": "讲一个冷笑话"}
]

response = chat_client.chat_completion(messages)
if response:
    print("AI回复:", response)

生成 2 个嵌入向量，每个维度为 1024
AI回复: 好的，接下来是一个冷笑话，希望你会喜欢：

世界上有两种动物，一种可以把自己藏在自己的壳里，一种却不能。猜猜看，哪种动物可以？
—— 答案：所有的动物都可以把自己藏在自己的壳里，除了裸鼹鼠（或者其他裸露的动物）。这个笑话的关键在于“壳”的双关，既指蜗牛、乌龟等有壳的动物，也指裸鼹鼠没有“壳”。


In [55]:
api_key = "sk-aa9a0b74449d42c3910391d753b5ae68"
base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"


emb_client = OpenAIEmbeddings(
    api_key=api_key,
    base_url=base_url,
    model="text-embedding-v2",
    check_embedding_ctx_length=False 
)

vectorstore = Chroma.from_documents(
    documents=doc_splits[:3],
    collection_name="rag-chroma",
    embedding=emb_client,
)
retriever = vectorstore.as_retriever()

In [42]:

chat_client = ChatOpenAI(
    api_key = api_key,
    base_url= base_url,
    model="qwen-plus" 
)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "你是谁？"}]
response = chat_client.invoke(messages)
print(response.json())

structured_llm_router = chat_client.with_structured_output(RouteQuery)

{"content":"我是通义千问，由阿里云研发的超大规模语言模型。我能够回答问题、创作文字，如写故事、公文、邮件、剧本等，还能进行逻辑推理、编程，甚至表达观点和玩游戏。我支持多种语言，包括中文、英文、德语、法语、西班牙语等。如果你有任何问题或需要帮助，欢迎随时告诉我！","additional_kwargs":{"refusal":null},"response_metadata":{"token_usage":{"completion_tokens":80,"prompt_tokens":22,"total_tokens":102,"completion_tokens_details":null,"prompt_tokens_details":{"audio_tokens":null,"cached_tokens":0}},"model_name":"qwen-plus","system_fingerprint":null,"id":"chatcmpl-bc0b677e-8086-4e8c-a5fc-06c84d4ec475","service_tier":null,"finish_reason":"stop","logprobs":null},"type":"ai","name":null,"id":"run--0e782be5-9ae2-4a64-aa2b-2004a057fb1c-0","example":false,"tool_calls":[],"invalid_tool_calls":[],"usage_metadata":{"input_tokens":22,"output_tokens":80,"total_tokens":102,"input_token_details":{"cache_read":0},"output_token_details":{}}}


### 1.build index

In [15]:
### Build Index

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

### from langchain_cohere import CohereEmbeddings

# Set embeddings
# embd = OpenAIEmbeddings()

# Docs to index
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=200, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)
doc_splits[0]

Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final resul

In [10]:
type(doc_splits)

list

In [20]:
# Add to vectorstore
# vectorstore = Chroma.from_documents(
#     documents=doc_splits[:3],
#     collection_name="rag-chroma",
#     embedding=emb_client,
# )
# retriever = vectorstore.as_retriever()


vectorstore = FAISS.from_documents(documents=doc_splits[:3],
                                   embedding=emb_client)
retriever = vectorstore.as_retriever()

### LLM

In [21]:
### Router

from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI


# Data model
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["vectorstore", "web_search"] = Field(
        ...,
        description="Given a user question choose to route it to web search or a vectorstore.",
    )


# LLM with function call
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm_router = chat_client.with_structured_output(RouteQuery)

# Prompt
system = """You are an expert at routing a user question to a vectorstore or web search.
The vectorstore contains documents related to agents, prompt engineering, and adversarial attacks.
Use the vectorstore for questions on these topics. Otherwise, use web-search."""
route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router
print(
    question_router.invoke(
        {"question": "Who will the Bears draft first in the NFL draft?"}
    )
)
print(question_router.invoke({"question": "What are the types of agent memory?"}))

AttributeError: 'SiliconFlowChat' object has no attribute 'with_structured_output'

## 1.文档分割

- langchain关于文档分割，提供了两个接口 CharacterTextSplitter &  RecursiveCharacterTextSplitter
- 不同的接口实现了不同的切割方式

### 基于长度的分割类型：CharacterTextSplitter
- token based：根据模型计算的token数量拆分文本，在使用语言模型时很有用。
- character based： 根据字符数拆分文本，这可以使不同类型的文本更加一致。

In [4]:
documents[0]

Document(metadata={'source': '日本', 'row': 0}, page_content='编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293')

In [5]:
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=2, chunk_overlap=1)
texts = text_splitter.split_text(documents[0].page_content)  # 接受str
len(texts), texts

(1,
 ['编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293'])

### 基于文本结构哦的分割类型：RecursiveCharacterTextSplitter
- 文本自然地被组织成段落、句子和单词等层级单元。利用这种固有结构（如分隔符，长度等）来指导我们的拆分策略。

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=40, chunk_overlap=20, separators=["\n\n", "\n", "。", "，", " ", ""])
docs = text_splitter.split_documents(documents)
docs

[Document(metadata={'source': '日本', 'row': 0}, page_content='编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki'),
 Document(metadata={'source': '日本', 'row': 0}, page_content='主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...'),
 Document(metadata={'source': '日本', 'row': 0}, page_content='年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险'),
 Document(metadata={'source': '日本', 'row': 0}, page_content='国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='编号: 31\n电影名称: 天堂电影院'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='导演: 朱塞佩·托纳多雷 Giuseppe Tornatore'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='主演: 莱昂纳多·迪卡普里奥 Leonardo...\n年份: 1988'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='年份: 1988\n国家: 意大利 法国\n分类: 剧情 爱情'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='分类: 剧情 爱情\n评分人数: 397297'),
 Document(metadata={'source': '日本', 'row': 2}, page_content='编号: 140\n电影名称: 萤火虫之墓'),
 Document(metadata={'source': '

In [7]:
from config import config
from llm_utils import SiliconFlowEmbedding, SiliconFlowChat

In [8]:
# 测试嵌入模型
emb_client = SiliconFlowEmbedding(api_key=config.llm_api_key,
                                           emb_model= config.emb_model,
                                           emb_api_url=config.emb_api_base)

test_texts = ["这是一个测试文档", "这是另一个测试文档"]
embeddings = emb_client.embed_documents(test_texts)
print(f"生成 {len(embeddings)} 个嵌入向量，每个维度为 {len(embeddings[0])}")

# 测试对话功能
chat_client = SiliconFlowChat(api_key=config.llm_api_key,
                                           chat_model= config.llm_model,
                                           chat_api_url= config.llm_api_base
                              )
messages = [
    {"role": "system", "content": "你是一个专业的AI助手。"},
    {"role": "user", "content": "讲一个冷效果"}
]

response = chat_client.chat_completion(messages)
if response:
    print("AI回复:", response)

生成 2 个嵌入向量，每个维度为 1024
AI回复: 好的，这里有一个冷笑话，希望能给你带来一丝清凉的感觉：

为什么电脑永远不会感冒？

因为它有“Windows”（窗户），但它是关闭的（关闭的窗户不容易让冷风或病毒进入）。

希望这个笑话能让你会心一笑，感受到一丝轻松和愉快。


## 2.向量库加载

In [9]:
vector_save_path = 'VectorStores/test_storage'
if not os.path.exists(vector_save_path):
    vector = FAISS.from_documents(documents, emb_client)
    vector.save_local(vector_save_path)
else:
    vector = FAISS.load_local(folder_path=vector_save_path, embeddings=emb_client,
                              allow_dangerous_deserialization=True)

## 3.LLM Chat Model定义

In [10]:
PROMPT_TEMPLATE = dict(
    RAG_PROMPT_TEMPALTE="""结合以上下文来回答用户的问题。
        问题: {question}
        可参考的上下文：
        ···
        {context}
        ···
        如果给定的上下文无法让你做出回答，请回答数据库中没有这个内容，不要臆想推测，请使用中文回答。
        回答:""",
)

## 4.RAG问答

In [11]:
query = '推荐一部宫崎骏的电影'
contents = vector.similarity_search_with_score(query, k=3)
print(f"检索结果： {contents}")
context = [c[0].page_content for c in contents]
print(f"检索结果文本： {context}")

检索结果： [(Document(id='4fe0e5fd-70f2-46c0-b7ba-e633cc271be5', metadata={'source': '日本', 'row': 0}, page_content='编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293'), np.float32(1.0679572)), (Document(id='d1d80a08-00e9-4edb-8367-54930f48a2da', metadata={'source': '日本', 'row': 2}, page_content='编号: 140\n电影名称: 萤火虫之墓\n导演: 高畑勋 Isao Takahata\n主演: 法拉赫阿米尔·哈什米安 Amir Fa...\n年份: 1988\n国家: 日本\n分类: 动画 剧情 战争\n评分人数: 257156'), np.float32(1.3351365)), (Document(id='ca566466-dd0b-4695-abda-046a6f176de1', metadata={'source': '意大利 法国', 'row': 1}, page_content='编号: 31\n电影名称: 天堂电影院\n导演: 朱塞佩·托纳多雷 Giuseppe Tornatore\n主演: 莱昂纳多·迪卡普里奥 Leonardo...\n年份: 1988\n国家: 意大利 法国\n分类: 剧情 爱情\n评分人数: 397297'), np.float32(1.4373972))]
检索结果文本： ['编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293', '编号: 140\n电影名称: 萤火虫之墓\n导演: 高畑勋 Isao Takahata\n主演: 法拉赫阿米尔·哈什米安 Amir Fa...\n年份: 1988\n国家: 日本

In [12]:
content = PROMPT_TEMPLATE['RAG_PROMPT_TEMPALTE'].format(question=query, context=context)
print(content)

messages = [
    {"role": "system", "content": "你是一个专业的AI助手。"},
    {"role": "user", "content": content}
]
response = chat_client.chat_completion(messages)
response

结合以上下文来回答用户的问题。
        问题: 推荐一部宫崎骏的电影
        可参考的上下文：
        ···
        ['编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293', '编号: 140\n电影名称: 萤火虫之墓\n导演: 高畑勋 Isao Takahata\n主演: 法拉赫阿米尔·哈什米安 Amir Fa...\n年份: 1988\n国家: 日本\n分类: 动画 剧情 战争\n评分人数: 257156', '编号: 31\n电影名称: 天堂电影院\n导演: 朱塞佩·托纳多雷 Giuseppe Tornatore\n主演: 莱昂纳多·迪卡普里奥 Leonardo...\n年份: 1988\n国家: 意大利 法国\n分类: 剧情 爱情\n评分人数: 397297']
        ···
        如果给定的上下文无法让你做出回答，请回答数据库中没有这个内容，不要臆想推测，请使用中文回答。
        回答:


'推荐您观看宫崎骏导演的电影《龙猫》。这部电影在1988年于日本上映，是一部动画奇幻冒险电影。'